In [22]:
import pandas as pd
import re
from pathlib import Path

from reportlab.lib import colors
from reportlab.lib.pagesizes import A3, landscape
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

In [23]:
from datetime import timedelta, date
import pandas as pd
import re
from pathlib import Path

from reportlab.lib import colors
from reportlab.lib.pagesizes import A3, landscape
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# =========================
# TWICE MONTHLY COMPANIES
# =========================
TWICE_MONTHLY_COMPANIES = [
    "БКС ДОЛНИ ЧИФЛИК ЕООД",
    "МЕРКУРИЙ ПРОИЗВОДСТВО И ПАКЕТАЖ АД",
    "СПЕДСТРОЙ ООД",
    "ФУУДС ТРЕЙД ЕООД",
    "ОП ДЕЗИНФЕКЦИЯ, ДЕЗИНСЕКЦИЯ, ДЕРАТИЗАЦИЯ",
    "ОП КОМПЛЕКС ЗА ДЕТСКО ХРАНЕНЕ",
    "КСУДПЛУ СВ. ЙОАН ЗЛАТОУСТ",
    "РАЙОН АСПАРУХОВО",
    "ОП ИНВЕСТИЦИОННА ПОЛИТИКА",
    "OП ТАСРУД",
    "ОП ЗООПАРК-СЦ ВАРНА",
    "ОП СПОРТ-ВАРНА",
    "РАЙОН МЛАДОСТ - ОБЩИНА ВАРНА",
    "РАЙОН ВЛАДИСЛАВ ВАРНЕНЧИК - ОБЩИНА ВАРНА",
    "КСУБЛС",
    "КСУДС",
    "КМЕТСТВО КАМЕНАР",
    "КМЕТСТВО КОНСТАНТИНОВО",
    "КМЕТСТВО ТОПОЛИ",
    "КСУВХ ГЕРГАНА ДСХ И ДПЛФУ",
    "КСУДМ",
    "ОП УПРАВЛЕНИЕ НА ПРОЕКТИ И ОЗЕЛЕНЯВАНЕ",
    "ДОМАШЕН СОЦИАЛЕН ПАТРОНАЖ",
    "РАЙОН ПРИМОРСКИ – ОБЩИНА ВАРНА",
    "ОБЩИНА ВАРНА - ДИРЕКЦИЯ СПОРТ",
    "ОБЩИНА ВАРНА - КМЕТСТВО КАЗАШКО",
    "ОБЩИНА ВАРНА % 1 - АВТОПАРК",
    "ОБЩИНА ВАРНА % 2 - ОБЩИНСКА ПОЛИЦИЯ",
    "ОБЩИНА ВАРНА % 3 - РАЗХОД ПО ПРОЕКТ ОИЦ ВАРНА",
    "ОБЩИНА ВАРНА % 4 - УЧИЛИЩЕН АВТОБУС",
    "КМЕТСТВО ЗВЕЗДИЦА",
    "ОП ОБЩИНСКИ ПАРКИНГИ И СИНЯ ЗОНА",
    "ОБЩИНА ВАРНА % 5 - ПДГ",
    "ОБЩИНА ВАРНА % 6 - УА",
    "РАЙОН ОДЕСОС",
    "ОБЩИНА АКСАКОВО % 1",
    "ОБЩИНА АКСАКОВО % 2"
]

# =========================
# SETTINGS
# =========================

INPUT_FILE = "../A_FINAL_RESULT/eko_transactions.xlsx"
OUTPUT_FOLDER = "RESULT/companies_result"

PDF_FONT_NAME = "Arial"
PDF_FONT_PATH = "C:/Windows/Fonts/arial.ttf"

# Реалните колони от Excel файла
PRODUCT_COLUMN = "Име на артикул"
LITERS_COLUMN = "Литри"
CARD_COLUMN = "Номер на карта"
GTA_AMOUNT_COLUMN = "Сума по GTA цена"
EKO_AMOUNT_COLUMN = "Сума по ЕКО цена"
DATE_COLUMNS = ["Дата", "Date"]

EXCLUDE_COLUMNS = ["Отстъпка", "Марж", "Печалба"]

# Колоните започват от ред 3 в Excel.
# В pandas това е header=2, защото броенето започва от 0.
EXCEL_HEADER_ROW = 2


# =========================
# HELPER FUNCTIONS
# =========================
def get_existing_date_column(df):
    for col in DATE_COLUMNS:
        if col in df.columns:
            return col

    raise ValueError(
        f"Не мога да намеря колона за дата. "
        f"Търсени колони: {DATE_COLUMNS}. "
        f"Налични колони: {df.columns.tolist()}"
    )

def get_invoice_period(current_date):
    """
    Връща периода за фактуриране според текущата дата.

    Ако сме от 1 до 15 число:
        фактурираме 16 - последен ден на предходния месец

    Ако сме от 16 до края на месеца:
        фактурираме 1 - 15 число на текущия месец
    """
    if current_date.day <= 15:
        first_day_current_month = current_date.replace(day=1)
        last_day_previous_month = first_day_current_month - timedelta(days=1)

        start_date = last_day_previous_month.replace(day=16)
        end_date = last_day_previous_month

    else:
        start_date = current_date.replace(day=1)
        end_date = current_date.replace(day=15)

    return start_date, end_date


def filter_twice_monthly_company(df, date_column, start_date, end_date):
    """
    Филтрира транзакциите само за конкретния период.
    """

    df = df.copy()

    df[date_column] = pd.to_datetime(
        df[date_column],
        errors="coerce",
    )

    df = df[
        (df[date_column].dt.date >= start_date) &
        (df[date_column].dt.date <= end_date)
    ]

    return df

def safe_folder_name(name):
    """
    Прави безопасно име за папка/файл.
    """
    name = str(name).strip()
    name = re.sub(r'[\\/*?:"<>|]', "_", name)
    name = re.sub(r"\s+", " ", name)
    return name[:80]

def get_company_name_from_sheet(input_path, sheet_name):
    """
    Взема пълното име на дружеството от първия ред на sheet-а.
    Ако не намери име, връща името на sheet-а.
    """

    preview_df = pd.read_excel(
        input_path,
        sheet_name=sheet_name,
        header=None,
        nrows=3
    )

    for row_idx in range(len(preview_df)):
        for value in preview_df.iloc[row_idx].dropna():
            value = str(value).strip()

            if "Наименование на дружеството" in value:
                company_name = (
                    value
                    .replace("Наименование на дружеството:", "")
                    .replace("Наименование на дружеството", "")
                    .replace(":", "")
                    .strip()
                )

                return company_name

    return sheet_name

def register_pdf_font():
    """
    Регистрира Unicode font за PDF, за да се показва кирилица правилно.
    """
    if not Path(PDF_FONT_PATH).exists():
        raise FileNotFoundError(f"PDF font not found: {PDF_FONT_PATH}")

    pdfmetrics.registerFont(TTFont(PDF_FONT_NAME, PDF_FONT_PATH))


def create_summary(df, product_col, liters_col):
    """
    Създава summary таблица:
    - продукт
    - общи литри
    - брой транзакции
    - средна претеглена GTA цена
    - сума по GTA
    - сума по ЕКО
    - разлика ЕКО - GTA
    - последен ред Общо
    """

    temp_df = df.copy()

    # Премахваме редове без продукт
    temp_df = temp_df.dropna(subset=[product_col])

    # Премахваме празни продукти като текст
    temp_df[product_col] = temp_df[product_col].astype(str).str.strip()
    temp_df = temp_df[temp_df[product_col] != ""]

    # Конвертираме нужните числови колони
    temp_df[liters_col] = pd.to_numeric(
        temp_df[liters_col],
        errors="coerce"
    ).fillna(0)

    temp_df[GTA_AMOUNT_COLUMN] = pd.to_numeric(
        temp_df[GTA_AMOUNT_COLUMN],
        errors="coerce"
    ).fillna(0)

    temp_df[EKO_AMOUNT_COLUMN] = pd.to_numeric(
        temp_df[EKO_AMOUNT_COLUMN],
        errors="coerce"
    ).fillna(0)

    summary = (
        temp_df
        .groupby(product_col)
        .agg(
            total_liters=(liters_col, "sum"),
            transactions_count=(liters_col, "count"),
            total_gta_amount=(GTA_AMOUNT_COLUMN, "sum"),
            total_eko_amount=(EKO_AMOUNT_COLUMN, "sum")
        )
        .reset_index()
    )

    # Средна претеглена GTA цена = Сума по GTA / Общо литри
    summary["weighted_gta_price"] = summary.apply(
        lambda row: row["total_gta_amount"] / row["total_liters"]
        if row["total_liters"] != 0 else 0,
        axis=1
    )



    summary = summary.sort_values(by="total_liters", ascending=False)

    summary = summary.rename(columns={
        product_col: "Продукт",
        "total_liters": "Общо / литри",
        "transactions_count": "Брой транзакции",
        "weighted_gta_price": "Средна претеглена GTA цена",
        "total_gta_amount": "Сума по GTA цена",
        "total_eko_amount": "Сума по ЕКО цена",
    })

    summary = summary[
        [
            "Продукт",
            "Общо / литри",
            "Брой транзакции",
            "Средна претеглена GTA цена",
            "Сума по GTA цена",
            "Сума по ЕКО цена",
        ]
    ]

    # Закръгляме числата
    summary["Общо / литри"] = summary["Общо / литри"].round(3)
    summary["Средна претеглена GTA цена"] = summary["Средна претеглена GTA цена"].round(4)
    summary["Сума по GTA цена"] = summary["Сума по GTA цена"].round(2)
    summary["Сума по ЕКО цена"] = summary["Сума по ЕКО цена"].round(2)

    # =========================
    # TOTAL ROW
    # =========================

    total_liters = summary["Общо / литри"].sum()
    total_gta_amount = summary["Сума по GTA цена"].sum()
    total_eko_amount = summary["Сума по ЕКО цена"].sum()

    total_weighted_gta_price = (
        total_gta_amount / total_liters
        if total_liters != 0 else 0
    )

    total_row = {
        "Продукт": "Общо",
        "Общо / литри": round(total_liters, 3),
        "Брой транзакции": int(summary["Брой транзакции"].sum()),
        "Средна претеглена GTA цена": "",
        "Сума по GTA цена": round(total_gta_amount, 2),
        "Сума по ЕКО цена": round(total_eko_amount, 2),
    }

    summary = pd.concat(
        [summary, pd.DataFrame([total_row])],
        ignore_index=True
    )

    # Защита срещу NaN / INF
    summary = summary.replace([float("inf"), float("-inf")], 0)
    summary = summary.fillna("")

    return summary


def write_company_excel(df, summary_df, output_path, company_name):
    """
    Създава Excel файл:
    - първо основните данни
    - след тях summary таблица
    """
    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        sheet_name = f"{company_name}"

        df.to_excel(writer, sheet_name=sheet_name, index=False, startrow=0)

        workbook = writer.book
        worksheet = writer.sheets[sheet_name]

        header_format = workbook.add_format({
            "bold": True,
            "bg_color": "#D9EAF7",
            "border": 1,
            "align": "center",
            "valign": "vcenter"
        })

        summary_title_format = workbook.add_format({
            "bold": True,
            "font_size": 14,
            "bg_color": "#BDD7EE",
            "border": 1,
            "align": "center"
        })

        summary_header_format = workbook.add_format({
            "bold": True,
            "bg_color": "#E2F0D9",
            "border": 1,
            "align": "center"
        })

        number_format = workbook.add_format({
            "num_format": "#,##0.00",
            "border": 1
        })

        main_number_format = workbook.add_format({
            "num_format": "#,##0.00"
        })

        weighted_price_format = workbook.add_format({
            "num_format": "#,##0.0000",
            "border": 1
        })

        text_format = workbook.add_format({
            "border": 1
        })

        total_text_format = workbook.add_format({
            "bold": True,
            "bg_color": "#ffff66",
            "border": 1
        })

        total_number_format = workbook.add_format({
            "bold": True,
            "bg_color": "#ffff66",
            "border": 1,
            "num_format": "#,##0.00"
        })



        # Основни headers
        for col_num, value in enumerate(df.columns):
            worksheet.write(0, col_num, value, header_format)

        # Auto width за основните данни
        for idx, column in enumerate(df.columns):
            column_data = df.iloc[:, idx].fillna("").astype(str)

            max_len = max(
                column_data.map(lambda value: len(str(value))).max() if not df.empty else 0,
                len(str(column))
            )

            if str(column).strip() == CARD_COLUMN:
                card_text_format = workbook.add_format({
                    "num_format": "@",
                })

                worksheet.set_column(idx, idx, min(max_len + 2, 35), card_text_format)
            else:
                worksheet.set_column(idx, idx, min(max_len + 2, 35))

        main_number_columns = [
            "Литри",
            "GTA цена",
            "ЕКО цена",
            "Сума по GTA цена",
            "Сума по ЕКО цена",
        ]

        for col_idx, column_name in enumerate(df.columns):
            if column_name in main_number_columns:
                worksheet.set_column(col_idx, col_idx, 15, main_number_format)


        # Summary започва няколко реда под основните данни
        summary_start_row = len(df) + 3

        worksheet.merge_range(
            summary_start_row,
            0,
            summary_start_row,
            len(summary_df.columns) - 1,
            f"Общи литри по продукт / {company_name}",
            summary_title_format
        )

        # Summary headers
        for col_num, value in enumerate(summary_df.columns):
            worksheet.write(
                summary_start_row + 1,
                col_num,
                value,
                summary_header_format
            )

        # Summary data
        for row_idx, row in summary_df.iterrows():
            excel_row = summary_start_row + 2 + row_idx

            is_total_row = str(row["Продукт"]).strip() == "Общо"

            for col_idx, column_name in enumerate(summary_df.columns):
                value = row[column_name]

                if pd.isna(value):
                    value = ""

                if column_name == "Продукт":
                    current_format = total_text_format if is_total_row else text_format
                    worksheet.write(excel_row, col_idx, str(value), current_format)
                else:
                    if column_name == "Средна претеглена GTA цена":
                        current_format = weighted_price_format
                    else:
                        current_format = total_number_format if is_total_row else number_format

                    if value == "":
                        worksheet.write(excel_row, col_idx, "", current_format)
                    else:
                        worksheet.write(excel_row, col_idx, float(value), current_format)

        worksheet.set_column(0, 0, 35)  # Продукт
        worksheet.set_column(1, 1, 18)  # Общо / литри
        worksheet.set_column(2, 2, 20)  # Брой транзакции
        worksheet.set_column(3, 3, 28)  # Средна претеглена GTA цена
        worksheet.set_column(4, 4, 22)  # Сума по GTA
        worksheet.set_column(5, 5, 18)  # Сума по ЕКО
        worksheet.set_column(7, 7, 23)  # Сума по ЕКО - основна справка
        worksheet.set_column(9, 9, 23)  # Сума по ЕКО - основна справка

        worksheet.freeze_panes(1, 0)




def prepare_pdf_table_data(df, max_text_len=35):
    """
    Подготвя dataframe за PDF таблица.
    Реже твърде дълги текстове.
    """
    clean_df = df.copy()

    for col in clean_df.columns:
        clean_df[col] = clean_df[col].fillna("").astype(str)
        clean_df[col] = clean_df[col].apply(
            lambda x: x[:max_text_len] + "..." if len(x) > max_text_len else x
        )

    return [list(clean_df.columns)] + clean_df.values.tolist()


def write_company_pdf(df, summary_df, output_path, company_name):
    """
    Създава PDF файл със:
    - име на фирмата
    - основните данни
    - summary с общи литри по продукт
    """

    register_pdf_font()

    doc = SimpleDocTemplate(
        str(output_path),
        pagesize=landscape(A3),
        rightMargin=1 * cm,
        leftMargin=1 * cm,
        topMargin=1 * cm,
        bottomMargin=1 * cm
    )

    styles = getSampleStyleSheet()

    styles["Title"].fontName = PDF_FONT_NAME
    styles["Heading1"].fontName = PDF_FONT_NAME
    styles["Normal"].fontName = PDF_FONT_NAME

    elements = []

    title = Paragraph(f"<b>{company_name}</b>", styles["Title"])
    elements.append(title)
    elements.append(Spacer(1, 0.5 * cm))

    # Основна таблица
    data_for_pdf = prepare_pdf_table_data(df, max_text_len=80)

    table = Table(data_for_pdf, repeatRows=1)

    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightblue),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),

        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),

        ("FONTNAME", (0, 0), (-1, -1), PDF_FONT_NAME),
        ("FONTSIZE", (0, 0), (-1, -1), 5),

        ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.whitesmoke]),
    ]))

    elements.append(table)
    elements.append(PageBreak())

    # Summary таблица
    summary_title = Paragraph(
        f"<b>ОБЩО ЛИТРИ ПО ПРОДУКТИ / {company_name}</b>",
        styles["Title"]
    )
    elements.append(summary_title)
    elements.append(Spacer(1, 0.4 * cm))

    summary_for_pdf = [list(summary_df.columns)] + summary_df.values.tolist()

    summary_table = Table(summary_for_pdf, repeatRows=1)

    summary_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgreen),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),

        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),

        ("FONTNAME", (0, 0), (-1, -1), PDF_FONT_NAME),
        ("FONTSIZE", (0, 0), (-1, -1), 9),

        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.whitesmoke]),
    ]))

    elements.append(summary_table)

    doc.build(elements)


def execute_pipeline():
    """
    Основна функция:
    - чете всички sheet-ове
    - създава папка за всяка фирма
    - създава Excel файл
    - създава PDF файл
    """
    input_path = Path(INPUT_FILE)
    output_root = Path(OUTPUT_FOLDER)

    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path.resolve()}")

    output_root.mkdir(parents=True, exist_ok=True)

    excel_file = pd.ExcelFile(input_path)

    processed_companies = []

    for sheet_name in excel_file.sheet_names:
        print(f"Processing sheet: {sheet_name}")

        df = pd.read_excel(
            input_path,
            sheet_name=sheet_name,
            header=EXCEL_HEADER_ROW,
            converters={
                CARD_COLUMN: lambda value: "" if pd.isna(value) else str(value).strip()
            }
        )

        # Премахва напълно празни редове
        df = df.dropna(how="all")

        # Премахва напълно празни колони
        df = df.dropna(axis=1, how="all")

        # Премахва колони, които не искаме да участват в резултатните файлове
        df = df.drop(
            columns=[col for col in EXCLUDE_COLUMNS if col in df.columns],
            errors="ignore"
        )

        if CARD_COLUMN in df.columns:
            df[CARD_COLUMN] = (
                df[CARD_COLUMN]
                .fillna("")
                .astype(str)
                .str.strip()
                .str.replace(r"\.0$", "", regex=True)
                .replace("nan", "")
            )

        if df.empty:
            print(f"Skipped empty sheet: {sheet_name}")
            continue

        if PRODUCT_COLUMN not in df.columns:
            raise ValueError(
                f"Не мога да намеря колоната за продукт в sheet: {sheet_name}. "
                f"Търсена колона: {PRODUCT_COLUMN}. "
                f"Налични колони: {df.columns.tolist()}"
            )

        if LITERS_COLUMN not in df.columns:
            raise ValueError(
                f"Не мога да намеря колоната за литри в sheet: {sheet_name}. "
                f"Търсена колона: {LITERS_COLUMN}. "
                f"Налични колони: {df.columns.tolist()}"
            )

        company_display_name = get_company_name_from_sheet(input_path, sheet_name)
        if company_display_name in TWICE_MONTHLY_COMPANIES:
            start_date, end_date = get_invoice_period(date.today())

            df = filter_twice_monthly_company(
                df=df,
                date_column=get_existing_date_column(df),
                start_date=start_date,
                end_date=end_date
            )

            print(f"Filtered period for {company_display_name}: {start_date} - {end_date}")

        if df.empty:
            print(f"Skipped company with no transactions in period: {company_display_name}")
            continue


        company_name = safe_folder_name(company_display_name)

        company_folder = output_root / company_name
        company_folder.mkdir(parents=True, exist_ok=True)

        summary_df = create_summary(
            df=df,
            product_col=PRODUCT_COLUMN,
            liters_col=LITERS_COLUMN
        )

        excel_output_path = company_folder / f"{company_name}.xlsx"
        pdf_output_path = company_folder / f"{company_name}.pdf"

        write_company_excel(
            df=df,
            summary_df=summary_df,
            output_path=excel_output_path,
            company_name=company_display_name
        )

        write_company_pdf(
            df=df,
            summary_df=summary_df,
            output_path=pdf_output_path,
            company_name=company_display_name
        )

        processed_companies.append(company_name)



    print("DONE")
    print(f"Processed companies: {len(processed_companies)}")
    print(f"Output folder: {output_root.resolve()}")


In [24]:
# =========================
# RUN
# =========================

execute_pipeline()

Processing sheet: OП ТАСРУД
Filtered period for OП ТАСРУД: 2026-05-16 - 2026-05-31
Skipped company with no transactions in period: OП ТАСРУД
Processing sheet: АГРАРЕН УНИВЕРСИТЕТ - ПЛОВДИВ
Processing sheet: АГРОПАКТ ЕООД
Processing sheet: АДИГА ЕООД
Processing sheet: АНИВА ЕООД
Processing sheet: БКС ДОЛНИ ЧИФЛИК ЕООД
Filtered period for БКС ДОЛНИ ЧИФЛИК ЕООД: 2026-05-16 - 2026-05-31
Skipped company with no transactions in period: БКС ДОЛНИ ЧИФЛИК ЕООД
Processing sheet: ДЕЛОР 20 ЕООД
Processing sheet: ДЕМАКС АД % ГРУПА 1
Processing sheet: ДЕМАКС АД % ГРУПА 2
Processing sheet: ДЕМАКС ДИ ПИ АЙ АД % ГРУПА 1
Processing sheet: ДЕМАКС ДИ ПИ АЙ АД % ГРУПА 2
Processing sheet: ДЕМАКС ДИ ПИ АЙ АД % ГРУПА 3
Processing sheet: ДЖИ ТИ ЕЙ ПЕТРОЛИУМ
Processing sheet: ДИЛЯНА-02 ЕООД
Processing sheet: ДИМИТРОВ ЕКСПРЕС КАРГО ООД
Processing sheet: ДИЯНИС ВАРНА ООД
Processing sheet: ДОК ТРАНС ЕООД
Processing sheet: ДОМАШЕН СОЦИАЛЕН ПАТРОНАЖ
Filtered period for ДОМАШЕН СОЦИАЛЕН ПАТРОНАЖ: 2026-05-16 - 2026-05

InvalidWorksheetName: Excel worksheet name 'ЖЕЛЯЗКОВ ИНЖЕНЕРИНГ - ПРОИЗВОДСТВЕНИ СИСТЕМИ ЕООД' must be <= 31 chars.